# Spatial Tokenization

- Creator: Sebastian Birk (sebastian.birk@helmholtz-munich.de).
- Date of Creation: 17.05.2024
- Date of Last Modification: 12.08.2025

## 1. Setup

### 1.1 Import Libraries

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
import os
import pickle
import warnings
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
import scanpy as sc
import scipy.sparse as sp
import scgpt as scg
import seaborn as sns
import squidpy as sq
import torch
from datasets import load_from_disk
from pyensembl import EnsemblRelease
from scipy.stats import spearmanr
from scgpt.model import TransformerModel
from scgpt.tokenizer.gene_tokenizer import GeneVocab
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder
from skmisc.loess import loess

from nichejepa import tokenizers
from nichejepa import utils
from nichejepa.models import get_1d_sincos_pos_embed_from_grid

ModuleNotFoundError: No module named 'scgpt'

### 1.2 Run Notebook Setup

In [ ]:
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

### 1.3 Configure Paths and Create Directories

In [ ]:
# Define paths
st_data_folder_path = "../../datasets/st_data" # spatially resolved transcriptomics data
st_data_bronze_folder_path = f"{st_data_folder_path}/bronze"
st_data_silver_folder_path = f"{st_data_folder_path}/silver"
st_data_gold_folder_path = f"{st_data_folder_path}/gold"
sc_data_folder_path = "../../datasets/sc_data" # spatially resolved transcriptomics data
sc_data_bronze_folder_path = f"{sc_data_folder_path}/bronze"
sc_data_silver_folder_path = f"{sc_data_folder_path}/silver"
sc_data_gold_folder_path = f"{sc_data_folder_path}/gold"
tokenizer = "cell_graph_tokenizer"
tokenizer_folder_path = f"../../datasets/tokenizer/{tokenizer}"
results_folder_path = "../../datasets/results"
geneformer_token_dictionary_file = f"../../datasets/tokenizer/geneformer_token_dictionary.pkl"
scgpt_model_dir = Path("../../artifacts/scgpt/pretrained_model")
scgpt_model_config_file = scgpt_model_dir / "args.json"
scgpt_model_file = scgpt_model_dir / "best_model.pt"
scgpt_vocab_file = scgpt_model_dir / "vocab.json"
scgpt_metadata_file = Path("../../artifacts/scgpt/metadata/gene_info.csv")

## 2. Tokenization

### 2.2 Compute Normalization Factors for Gene Counts

Corpus means, non-zero medians, or log means are used depending on the 'norm_method' specified in the CellGraphRankTokenizer (check source code).